# Homework 2 – Betti Numbers and Boundary Matrix Reduction

**Barthélémy:** NDECKY 
**Institution:** AIMS  
**Course:** Computational Topology  
**30/03/2026:** 

## Introduction

In this notebook, we implement two fundamental algorithms from computational topology:

- **Algorithm 1:** Incremental computation of Betti numbers
- **Algorithm 2:** Reduction of the boundary matrix (modulo 2)

These algorithms allow us to compute topological invariants (Betti numbers), which describe the number of connected components, cycles, and higher-dimensional holes in a simplicial complex.

 ### 1 Implement Algorithm 1 and Algorithm 2 in python.

## Algorithm 1: Incremental Betti Numbers

This algorithm processes simplices in increasing order and updates Betti numbers depending on whether a simplex creates or destroys a topological feature.

In [15]:
# Algorithm 1
def incremental_betti(complexes):
    """
    Implementation of Algorithm 1.
    complexes: list of tuples (dimension, is_positive)
    """
    if not complexes:
        return []
    
    max_dim = max(d for d, _ in complexes)
    betti = [0] * (max_dim + 1)

    for (d, positive) in complexes:
        if positive:
            betti[d] += 1
        else:
            if d > 0:
                betti[d-1] -= 1
                
    return betti

## Example: Triangulation of a Circle

We construct a triangle (3 vertices and 3 edges), which is topologically equivalent to a circle.

Expected result:
- β₀ = 1 (one connected component)
- β₁ = 1 (one cycle)

In [16]:
# Example: triangle (circle)
triangle = [
    (0, True),   # Vertex A
    (0, True),   # Vertex B
    (0, True),   # Vertex C
    (1, False),  # Edge AB
    (1, False),  # Edge BC
    (1, True)    # Edge CA
]

betti = incremental_betti(triangle)

print("Betti numbers:", betti)
print(f"beta_0 = {betti[0]}, beta_1 = {betti[1]}")

Betti numbers: [1, 1]
beta_0 = 1, beta_1 = 1


## Algorithm 2: Boundary Matrix Reduction

This algorithm reduces a boundary matrix using column operations modulo 2.

The goal is to identify pivots (lowest 1s) and eliminate duplicate pivots.

In [17]:
def reduce_boundary_matrix(A):
    n = len(A)
    
    # Copy matrix
    R = [[A[i][j] for j in range(n)] for i in range(n)]

    def get_column(j):
        return [R[i][j] for i in range(n)]

    def low(col):
        for i in reversed(range(len(col))):
            if col[i] == 1:
                return i
        return -1

    def add_columns(i, j):
        for row in range(n):
            R[row][j] = (R[row][j] + R[row][i]) % 2

    for j in range(n):
        while True:
            found = False
            col_j = get_column(j)
            for i in range(j):
                col_i = get_column(i)
                if low(col_i) == low(col_j) and low(col_j) != -1:
                    add_columns(i, j)
                    col_j = get_column(j)
                    found = True
                    break
            if not found:
                break

    return R

## Example: Boundary Matrix Reduction

In [18]:
A = [
    [1, 0, 1],
    [0, 1, 1],
    [0, 0, 1]
]

R = reduce_boundary_matrix(A)

print("Reduced matrix:")
for row in R:
    print(row)

Reduced matrix:
[1, 0, 1]
[0, 1, 1]
[0, 0, 1]


## 2 Termination of the Algorithms

In this section, we prove that both Algorithm 1 and Algorithm 2 terminate after a finite number of steps.

### Algorithm 1: Incremental Betti Numbers

**Claim:** The algorithm terminates after a finite number of steps.

**Proof:**

- The input is a finite sequence of simplices:
  
  K₁ ⊂ K₂ ⊂ ... ⊂ Kₙ

- The algorithm consists of a single loop:
  
  for i = 1 to n

- At each iteration:
  - A constant number of operations is performed:
    - checking whether the simplex is positive or negative
    - updating one Betti number

- There are no nested loops or recursive calls.

Therefore, the algorithm performs exactly **n iterations**, where n is the number of simplices.

Hence, Algorithm 1 terminates after a finite number of steps. ∎

### Algorithm 2: Boundary Matrix Reduction

**Claim:** The algorithm terminates after a finite number of steps.

**Proof:**

- The matrix has a finite number of columns.

- For each column j, the while loop performs column additions modulo 2.

- At each step:
  - Either the pivot δ(j) strictly decreases,
  - Or the column becomes zero.

- Since the number of rows is finite, δ(j) can take only finitely many values.

- Moreover, each column belongs to (ℤ/2ℤ)^n, which is a finite set.

Thus, the process cannot continue indefinitely.

Therefore, the algorithm terminates after a finite number of steps. ∎

## Question 3 (a): Triangulation of a Circle

We consider a simplicial complex corresponding to a triangulation of the circle \( S^1 \).

A simple model consists of:
- 3 vertices
- 3 edges forming a closed loop

---

### Incremental Construction (Algorithm 1)

We add the simplices in the following order:

- \( (0, $\text{True}$) \): vertex \( $v_0$ \) → \( $\beta_0 = 1$ \)
- \( (0, $\text{True}$) \): vertex \( $v_1$ \) → \( $\beta_0 = 2$ \)
- \( (0, $\text{True}$) \): vertex \( $v_2$\) → \( $\beta_0 = 3$ \)

- \( (1, $\text{False}$) \): edge \( $e_{01}$ \) → connects components → \( $\beta_0 = 2$ \)
- \( (1, $\text{False}$) \): edge \( $e_{12}$ \) → connects components → \( $\beta_0 = 1$ \)

- \( (1, $\text{True}$) \): edge \( $e_{20}$\) → closes a loop → \( $\beta_1 = 1$ \)

---

### Interpretation

- \( $\beta_0 = 1$ \): the complex is connected  
- \( $\beta_1 = 1$ \): there is one 1-dimensional hole (cycle)

---

### Conclusion

The computed Betti numbers are:
\[
$\beta_0 = 1$, $\quad \beta_1 = 1$
\]

This agrees with the topology of the circle \( $ S^1$ \). 

In [20]:
# Triangulation of a circle (triangle)

circle_complex = [
    (0, True),   # Vertex A
    (0, True),   # Vertex B
    (0, True),   # Vertex C
    (1, False),  # Edge AB (connects components)
    (1, False),  # Edge BC (connects components)
    (1, True)    # Edge CA (creates a cycle)
]

betti_circle = incremental_betti(circle_complex)

print("Betti numbers for the circle:", betti_circle)

Betti numbers for the circle: [1, 1]


## Link with Algorithm 2 (Boundary Matrix Reduction)

The classification of simplices as **positive** or **negative** in Algorithm 1
comes from the reduction of the boundary matrix in Algorithm 2.

---

### Boundary matrices

For the circle (triangle), we consider:

- \( $\partial_1$ \): boundary map from edges to vertices

Each column corresponds to an edge, and contains its boundary vertices (mod 2).

For example:

- \( $e_{01} \mapsto v_0 + v_1$ \)
- \( $e_{12} \mapsto v_1 + v_2$ \)
- \( $e_{20} \mapsto v_2 + v_0$ \)

This gives the boundary matrix:

$$
\partial_1 =
\begin{bmatrix}
1 & 0 & 1 \\
1 & 1 & 0 \\
0 & 1 & 1
\end{bmatrix}
$$

---

### Reduction (Algorithm 2)

We reduce the matrix column by column (mod 2):

- The first two columns have distinct pivots → they are **negative simplices**
  (they eliminate connected components, decreasing \( $\beta_0$ \)).

- The third column reduces to a column with no pivot conflict → it is a **positive simplex**.

---

### Interpretation

- Negative simplices correspond to columns with a pivot:
  → they "kill" a homology class

- Positive simplices correspond to columns without pivot:
  → they "create" a homology class
  

## Question 3 (b): Triangulation of a 2-dimensional Sphere \( S^2 \)

We model the sphere using the boundary of a tetrahedron, which is the simplest triangulation of \( S^2 \).

A simplicial decomposition consists of:
- 4 vertices
- 6 edges
- 4 triangular faces

---

### Incremental Construction (Algorithm 1)

We add simplices in the following order:

- \( (0, $\text{True}$) $\times 4$ \): vertices \( $v_0, v_1, v_2, v_3$\)  
  → \( $\beta_0 = 4$ \)

- \( (1, $\text{False}$) $\times 3$ \): edges forming a spanning tree  
  → \( $\beta_0 = 1$ \)

- \( (1, $\text{True}$) $\times 3$ \): remaining edges create cycles  
  → \( $\beta_1 = 3$ \)

- \( (2, $\text{False}$) $\times 3$ \): three faces fill these cycles  
  → \( $\beta_1 = 0$ \)

- \( (2, $\text{True}$) \): last face creates a 2-dimensional cavity  
  → \( $\beta_2 = 1$ \)

---

### Interpretation

- \( $\beta_0 = 1$ \): the sphere is connected  
- \( $\beta_1 = 0$ \): all cycles are filled by faces  
- \( $\beta_2 = 1$ \): one 2-dimensional cavity is enclosed  

---

### Conclusion

$$
\beta_0 = 1, \quad \beta_1 = 0, \quad \beta_2 = 1
$$

This matches the topology of the sphere \( S^2 \).

In [26]:
# (b) Sphere S^2 — correct minimal model

sphere_complex = [
    (0, True),   # Vertex A
    (0, True),   # Vertex B
    (0, True),   # Vertex C

    (1, False),  # Edge AB (connects)
    (1, False),  # Edge BC (connects)
    (1, True),   # Edge CA (creates a cycle)

    (2, False),  # Face fills the cycle → kills beta_1
    (2, True)    # Creates a 2D cavity → beta_2 = 1
]

betti_sphere = incremental_betti(sphere_complex)

print("Betti numbers for the sphere:", betti_sphere)
print(f"beta_0 = {betti_sphere[0]}, beta_1 = {betti_sphere[1]}, beta_2 = {betti_sphere[2]}")

Betti numbers for the sphere: [1, 0, 1]
beta_0 = 1, beta_1 = 0, beta_2 = 1


## Question 3 (c): Triangulation of a Torus \( T^2 \)

A torus can be obtained by identifying opposite edges of a square.

A standard triangulation leads to:
- 1 vertex (after identifications)
- 2 independent edges (generators of loops)
- 1 face

---

### Incremental Construction (Algorithm 1)

We describe the evolution of Betti numbers:

- \( (0, $\text{True}$) \): single vertex  
  → \( $\beta_0 = 1$ \)

- \( (1, $\text{True}$) $\times 2$ \): two independent edges  
  → each creates a cycle  
  → \( $\beta_1 = 2$ \)

- \( (2, $\text{True}$) \): one face  
  → creates a 2-dimensional cavity  
  → \( $\beta_2 = 1$ \)

---

### Interpretation

- \( $\beta_0 = 1$ \): one connected component  
- \( $\beta_1 = 2$ \): two independent cycles (the two fundamental loops of the torus)  
- \( $\beta_2 = 1$ \): one 2-dimensional cavity  

---

### Conclusion

$$
\beta_0 = 1, \quad \beta_1 = 2, \quad \beta_2 = 1
$$

This matches the topology of the torus \( T^2 \). 

In [27]:
# (c) Triangulation of a torus (minimal model)

torus_complex = [
    (0, True),   # Single vertex
    (1, True),   # First independent loop
    (1, True),   # Second independent loop
    (2, True)    # Face creates a 2D cavity
]

betti_torus = incremental_betti(torus_complex)

print("Betti numbers for the torus:", betti_torus)
print(f"beta_0 = {betti_torus[0]}, beta_1 = {betti_torus[1]}, beta_2 = {betti_torus[2]}")

Betti numbers for the torus: [1, 2, 1]
beta_0 = 1, beta_1 = 2, beta_2 = 1


## Question 3 (d)

We consider a triangle \([A,B,C]\) where all vertices are identified:
$$
A \sim B \sim C = p
$$

---

### Structure of the complex

After identification:
- 1 vertex
- 3 edges (each becomes a loop at the same point)
- 1 face

---

### Incremental Interpretation (Algorithm 1)

- \( (0, $\text{True}$) \): one vertex  
  → \( $\beta_0 = 1$ \)

- \( (1, $\text{True}$) $\times 3$ \): three loops  
  → \( $\beta_1 = 3$ \)

- \( (2, $\text{False}$) \): the face fills one relation between loops  
  → \( $\beta_1 = 2$ \)

---

### Interpretation

- \( $\beta_0 = 1$ \): one connected component  
- \( $\beta_1 = 2$ \): two independent cycles  

---

### Conclusion

$$
\beta_0 = 1, \quad \beta_1 = 2
$$

This space is equivalent to a wedge of two circles.

In [21]:
# (d) Triangle with all vertices identified

complex_d = [
    (0, True),   # Single vertex p
    (1, True),   # Edge e1 (loop)
    (1, True),   # Edge e2 (loop)
    (1, True),   # Edge e3 (loop)
    (2, False)   # Face removes one relation
]

betti_d = incremental_betti(complex_d)

print("Betti numbers for (d):", betti_d)
print(f"beta_0 = {betti_d[0]}, beta_1 = {betti_d[1]}")

Betti numbers for (d): [1, 2, 0]
beta_0 = 1, beta_1 = 2


## Question 3 (e)

We consider a triangle where edges are identified according to the given figure.

This construction typically produces a surface with non-trivial topology.

---

### Key idea

- Edge identifications create loops and relations between them
- The result depends on orientation:
  - can produce a torus
  - or a projective plane

---

### Standard case (torus-type identification)

- 1 vertex
- 2 independent edges
- 1 face

---

### Betti numbers

$$
\beta_0 = 1, \quad \beta_1 = 2, \quad \beta_2 = 1
$$

---

### Interpretation

- one connected component  
- two independent cycles  
- one 2D cavity   

In [22]:
# (e) Triangle with edge identifications (torus case)

complex_e = [
    (0, True),   # Single vertex
    (1, True),   # First independent loop
    (1, True),   # Second independent loop
    (2, True)    # Face creates a 2D cavity
]

betti_e = incremental_betti(complex_e)

print("Betti numbers for (e):", betti_e)
print(f"beta_0 = {betti_e[0]}, beta_1 = {betti_e[1]}, beta_2 = {betti_e[2]}")

Betti numbers for (e): [1, 2, 1]
beta_0 = 1, beta_1 = 2, beta_2 = 1
